# 🚀 Huấn luyện Mô hình ABSA bằng PhoBERT trên Kaggle

Sử dụng dữ liệu auto-labeled v2.

In [ ]:
!pip install transformers datasets evaluate pyvi
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
from sklearn.model_selection import train_test_split

# 1. Đường dẫn tới Dataset mới trên Kaggle
DATA_DIR = '/kaggle/input/datasets/quockhanhbe/shopping-reviews-v2/' 

print("Đang tải và gộp dữ liệu...")
df_pos = pd.read_csv(DATA_DIR + 'auto_labeled_cleaned_positive_reviews.csv')
df_neg = pd.read_csv(DATA_DIR + 'auto_labeled_cleaned_negative_reviews.csv')
df_total = pd.concat([df_pos, df_neg], ignore_index=True)

In [ ]:
# 2. Định nghĩa hàm biến đổi dữ liệu (NLI)
def flatten_dataset(df):
    aspects = {
        'Quality': 'chất lượng',
        'Price': 'giá cả',
        'Delivery': 'giao hàng',
        'Service': 'dịch vụ'
    }
    
    new_data = []
    for _, row in df.iterrows():
        comment = str(row['cleaned_comment'])
        for asp_key, asp_text in aspects.items():
            label = int(row[asp_key])
            if label == -1:
                label = 3
                
            new_data.append({
                'text': comment,
                'aspect': asp_text,
                'label': label
            })
    return pd.DataFrame(new_data)

print("Đang chuyển đổi sang định dạng NLI...")
df_nli = flatten_dataset(df_total)

In [ ]:
# 3. Chia tập Train (80%), Val (10%), Test (10%)
train_df, temp_df = train_test_split(df_nli, test_size=0.2, random_state=42, stratify=df_nli['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Chuyển đổi sang định dạng Hugging Face Dataset
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})

In [ ]:
# 4. Tokenize dữ liệu với PhoBERT
model_checkpoint = "vinai/phobert-base-v2"  # Dùng bản v2 mới nhất của VinAI
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples["aspect"], 
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

print("Đang Tokenize dữ liệu...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# 5. Định nghĩa Metrics
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

# 6. Tải Model PhoBERT
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)

In [ ]:
# 7. Cấu hình Tham số Huấn luyện
training_args = TrainingArguments(
    output_dir="/kaggle/working/phobert-absa",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True, # Bật tăng tốc GPU
    report_to="none" 
)

# 8. Khởi tạo Trainer và Huấn luyện
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("🚀 Bắt đầu huấn luyện...")
trainer.train()

In [ ]:
# 9. Đánh giá và Lưu
print("Đánh giá trên tập Test:")
print(trainer.evaluate(tokenized_datasets["test"]))

trainer.save_model("/kaggle/working/phobert-absa-final")
tokenizer.save_pretrained("/kaggle/working/phobert-absa-final")

# Nén file zip tải về
import shutil
shutil.make_archive('/kaggle/working/phobert-absa-final', 'zip', '/kaggle/working/phobert-absa-final')
print("Đã nén thành công phobert-absa-final.zip! Hãy tải về máy.")